# Step 1 — Enteric and manure emissions cost accounting

This notebook estimates country-year enteric fermentation and manure management damages by converting FAOSTAT emissions to tonnes CO$_2$e and applying low, mid, and high social cost of carbon (SCC) scenarios.

## Inputs and outputs

**Inputs**

- `../data/raw/FAOSTAT_emissions_2000-2022.csv`

**Outputs**

- `../data/outputs/enteric_manure_emissions_cost_country_year.csv`

Place the notebook in `code/` and keep the raw and output files in parallel `data/raw` and `data/outputs` folders.

## Notebook workflow

This notebook is organized into short executable sections so that each transformation step is visible in GitHub and easy for reviewers to follow.

## Analysis step

In [ ]:
import pandas as pd

EMIS_PATH = "./FAOSTAT_emissions_2000-2022.csv"
OUT_PATH  = "./enteric_manure_emissions_cost_country_year.csv"

emis = pd.read_csv(EMIS_PATH)

## Filter livestock process emissions

In [ ]:
emis_f = emis[
    (emis["Source"] == "FAO TIER 1") &
    (emis["Element"] == "Emissions (CO2eq) (AR5)") &
    (emis["Item"].isin(["Enteric Fermentation", "Manure Management"])) &
    (emis["Year"].between(2000, 2022))
].copy()

## Ensure numeric

In [ ]:
emis_f["Value"] = pd.to_numeric(emis_f["Value"], errors="coerce")
emis_f = emis_f.dropna(subset=["Value", "Unit"])

## Check units

In [ ]:
units = sorted(emis_f["Unit"].unique())
if len(units) != 1:
    raise ValueError(f"Multiple emissions units found: {units}. Please harmonize units first.")
unit = units[0]

## Convert to tonnes CO2e

In [ ]:
if unit == "kt":
    emis_f["tCO2e"] = emis_f["Value"] * 1_000.0
elif unit in ["t", "tonnes", "tonne"]:
    emis_f["tCO2e"] = emis_f["Value"].astype(float)
elif unit == "kg":
    emis_f["tCO2e"] = emis_f["Value"] / 1_000.0
else:
    raise ValueError(f"Unexpected emissions unit: {unit}")

## Aggregate to country-year

In [ ]:
emis_cty = (
    emis_f.groupby(["Area", "Area Code (M49)", "Year"], as_index=False)
          .agg(tCO2e_enteric_manure=("tCO2e", "sum"))
)

## SCC scenarios (constant real shadow prices)

In [ ]:
scc_params = pd.DataFrame([
    {"scenario": "low",  "SCC_usd_per_tCO2e": 50},
    {"scenario": "mid",  "SCC_usd_per_tCO2e": 100},
    {"scenario": "high", "SCC_usd_per_tCO2e": 200},
])

## Cross join

In [ ]:
emis_cty["key"] = 1
scc_params["key"] = 1
df = emis_cty.merge(scc_params, on="key").drop(columns="key")

# Monetize
df["enteric_manure_cost_usd"] = df["tCO2e_enteric_manure"] * df["SCC_usd_per_tCO2e"]

# Save
df.to_csv(OUT_PATH, index=False)
print("Saved file:", OUT_PATH)
print(df.head())